In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
os.environ.get("BIRDDOG_USE_LOCAL_NOCODB")

In [3]:
from birddog.database import Database
from birddog.wiki import (
    page_label,
    sequential_page_label,
)
from birddog.nocodb_database import (
    clone_table_schema,
    copy_records,
    rename_field,
)

2026-04-16 12:06:01,099 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-04-16 12:06:01,249 [INFO] Translation is enabled. Using GCP translator
2026-04-16 12:06:01,250 [INFO] Using Google Cloud translation API
2026-04-16 12:06:01,250 [INFO] GoogleCloudTranslator using REST API


In [4]:
db = Database()

2026-04-16 12:06:01,482 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     8.08    39.00       0.00           24


In [13]:
pages = db.get_all_ids("Pages")

2026-04-16 12:47:28,104 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   66.64     0.89    39.00       0.00           24


In [14]:
len(pages)

0

In [12]:
db.delete("Pages", pages)

2026-04-16 12:40:05,420 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   37.64     3.16    39.00       0.00           24
2026-04-16 12:41:05,633 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   43.64     4.30    39.00       0.00           24
2026-04-16 12:42:05,727 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   49.64     4.26    39.00       0.00           24
2026-04-16 12:43:05,764 [

13869

In [18]:
docs = db.get_all_ids("Documents")

2026-04-16 12:52:50,598 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   86.64     0.56    39.00       0.00           24


In [19]:
len(docs)

0

In [17]:
db.delete("Documents", docs)

2026-04-16 12:48:28,133 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   71.64     3.91    39.00       0.00           24
2026-04-16 12:49:28,308 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   77.64     4.37    39.00       0.00           24
2026-04-16 12:50:28,441 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   83.64     4.34    39.00       0.00           24


8286

In [ ]:
pages = []
cursor = None
while True:
    if cursor and (int(cursor) % 10000) == 0:
        print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Pages", 
        cursor=cursor, 
        limit=1000,
        where=("seq_label", "is", None),
        fields=("Id","title","label","seq_label"),
    )
    if batch:
        pages.extend(batch)
    if not cursor:
        break

In [ ]:
len(pages)

In [ ]:
pages[1000]

In [ ]:
def normalize_labels(page):
    result = page.copy()
    title = page.get("title")
    if title:
        proper_label = page_label(title)
        result["label"] = proper_label
        proper_seq_label = sequential_page_label(proper_label)
        result["seq_label"] = proper_seq_label
    return result

In [ ]:
normalize_labels(pages[2000])

In [ ]:
pages[2000]

In [ ]:
pages[2000] == normalize_labels(pages[2000])

In [ ]:
pages[2000] == pages[2000].copy()

In [ ]:
norm_pages = [normalize_labels(p) for p in pages]

In [ ]:
norm_pages[:10]

In [ ]:
changed_pages = [n for n,p in zip(norm_pages, pages) if n != p]

In [ ]:
len(changed_pages)

In [ ]:
len(pages)

In [ ]:
len(norm_pages)

In [ ]:
rec_ids = db.write("Pages", norm_pages[1000:2000])

In [ ]:
chunk = 1000
for i in range(0, len(norm_pages), chunk):
    print(i)
    rec_ids = db.write("Pages", norm_pages[i:(i+chunk)])